In [ ]:
from agents.mcp import MCPServerStdio
from agents import (
    Agent,
    Runner,
    input_guardrail,
    ModelSettings,
    RunContextWrapper,
    TResponseInputItem,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    set_trace_processors,
)
from mcp.types import TextContent
import os
import json
from agents.extensions.models.litellm_model import LitellmModel
from datetime import datetime, timedelta
import opik
from opik.integrations.openai.agents import OpikTracingProcessor
import litellm
from IPython.display import display, Markdown
from pydantic import BaseModel, Field

In [ ]:
MODEL_NAME = 'google/gemini-3-flash-preview'
LIGHT_MODEL_NAME = 'google/gemini-2.5-flash-lite'

low_reasoning_enabled = ModelSettings(
    extra_body={
        'reasoning': {
            'enabled': True,
            'effort': 'low',
        }
    }
)

medium_reasoning_enabled = ModelSettings(
    extra_body={
        'reasoning': {
            'enabled': True,
            'effort': 'medium',
        }
    }
)

litellm_model = LitellmModel(
    model='openrouter/' + MODEL_NAME,
    base_url='https://openrouter.ai/api/v1/',
    api_key=os.environ['OPENROUTER_KEY'],
)

light_model = LitellmModel(
    model='openrouter/' + LIGHT_MODEL_NAME,
    base_url='https://openrouter.ai/api/v1/',
    api_key=os.environ['OPENROUTER_KEY'],
)


def get_calendar_readonly_server():
    return MCPServerStdio(
        name='calendar',
        params={
            'command': 'npx',
            'args': ['@cocal/google-calendar-mcp'],
            'env': {
                'GOOGLE_OAUTH_CREDENTIALS': os.environ['GOOGLE_OAUTH_CREDENTIALS'],
                'ENABLED_TOOLS': (
                    'manage-accounts,list-calendars,list-events,'
                    'get-event,search-events,get-current-time'
                ),
            },
        },
        client_session_timeout_seconds=60,
    )


def get_calendar_full_access_server():
    return MCPServerStdio(
        name='calendar',
        params={
            'command': 'npx',
            'args': ['@cocal/google-calendar-mcp'],
            'env': {
                'GOOGLE_OAUTH_CREDENTIALS': os.environ['GOOGLE_OAUTH_CREDENTIALS'],
            },
        },
        client_session_timeout_seconds=60,
    )

### Check MCP server & authentication

In [ ]:
async with get_calendar_readonly_server() as server:
    tools = await server.list_tools()
    time_result = await server.call_tool('get-current-time', {})
    assert isinstance(time_result.content[0], TextContent)
    current_time = json.loads(time_result.content[0].text)
    current_datetime = datetime.strptime(current_time['currentTime'], '%Y-%m-%dT%H:%M:%S.%f%z')

    calendars_result = await server.call_tool('list-calendars', {})
    assert isinstance(calendars_result.content[0], TextContent)
    calendar_list = json.loads(calendars_result.content[0].text)['calendars']

    datetime_min = current_datetime - timedelta(days=7)
    datetime_max = current_datetime + timedelta(days=7)
    events_result = await server.call_tool(
        'list-events',
        {
            'calendarId': [calendar['id'] for calendar in calendar_list[:50]],
            'timeMin': datetime_min.strftime('%Y-%m-%dT%H:%M:%S'),
            'timeMax': datetime_max.strftime('%Y-%m-%dT%H:%M:%S'),
        },
    )
    assert isinstance(events_result.content[0], TextContent)
    events = json.loads(events_result.content[0].text)

In [4]:
{event['summary']: event['start']['dateTime'] for event in events['events'][1:3]}

{'Bouldering with Luca': '2026-02-11T15:30:00+08:00',
 'Chinese Trial Lesson': '2026-02-11T18:00:00+08:00'}

### Set up Opik tracking

In [3]:
opik.configure()
os.environ['OPIK_PROJECT_NAME'] = 'calendar-agent-1'
set_trace_processors(processors=[OpikTracingProcessor()])

OPIK: Opik is already configured. You can check the settings by viewing the config file at /Users/olegsmirnov/.opik.config
OPIK: Configuration completed successfully. Traces will be logged to 'Default Project' project. To change the destination project, see: https://www.comet.com/docs/opik/tracing/log_traces#configuring-the-project-name


In [4]:
litellm.suppress_debug_info = True

### Tool-augmented agent with reasoning

In [15]:
async with get_calendar_readonly_server() as server:
    calendar_agent = Agent(
        name='calendar-agent',
        instructions=(
            'You are a helpful calendar assistant. '
            'Always use the available tools to answer. '
            'When you need information, you should call tools explicitly, using special syntax. '
            'Never guess arguments, only pass arguments that you\'re certain about. '
            'When you\'re getting an error, probably, you made a mistake in tool calling. '
            'Always be comprehensive and check yourself. '
            'Please note that if a function returns an empty list, '
            'this might be because there\'s something wrong with its arguments. '
            'Note that events may be scattered across different calendars.'
        ),
        model=litellm_model,
        mcp_servers=[server],
        model_settings=low_reasoning_enabled,
    )
    result = await Runner.run(
        calendar_agent,
        'Which, when, and with whom did I do sports last week?',
        max_turns=30,
    )
    display(Markdown(result.final_output))

Last week (February 9th to February 15th, 2026), you had the following sports-related activities:

*   **Bouldering:** On Wednesday, February 11th, from 3:30 PM to 4:30 PM with **Luca**.
*   **Gym and Run:** On Thursday, February 12th, from 7:00 PM to 8:30 PM with **Luca**.

### Multi-agent system

In [5]:
class WorkoutPlannerInput(BaseModel):
    goal: str = Field(
        description=(
            'Any user\'s workout goals, e.g. "lose weight", '
            '"prepare for a marathon", "tennis competition", etc.'
        )
    )
    number_of_days: int = Field(
        description='Number of days for the workout plan. Should be a positive integer.'
    )
    preferences: str | None = Field(default=None, description='Any user preferences.')


workout_planner_agent = Agent(
    name='workout-planner-agent',
    instructions=(
        'You are a professional coach. '
        'You should come up with a workout plan based on the user\'s goals. '
        'You\'ll be given number of days, and goals, and you need to combine '
        'a workout program for this number of days. '
        'Assume that the athlete is a professional.'
    ),
    model=litellm_model,
    model_settings=medium_reasoning_enabled,
)

async with get_calendar_full_access_server() as server:
    calendar_agent = Agent(
        name='calendar-agent',
        instructions=(
            'You are a helpful calendar assistant. '
            'Always use the available tools to answer. '
            'When you need information, you should call tools explicitly, using special syntax. '
            'Never guess arguments, only pass arguments that you\'re certain about. '
            'When you\'re getting an error, probably, you made a mistake in tool calling. '
            'Always be comprehensive and check yourself. '
            'Please note that if a function returns an empty list, '
            'this might be because there\'s something wrong with its arguments. '
            'Note that events may be scattered across different calendars.'
        ),
        model=litellm_model,
        mcp_servers=[server],
        tools=[
            workout_planner_agent.as_tool(
                tool_name='workout-planner',
                tool_description=(
                    'Plans a workout program, given user\'s goals '
                    'and number of days, as a professional coach. '
                    'Should be called to create workout plans.'
                ),
                parameters=WorkoutPlannerInput,
            )
        ],
        model_settings=low_reasoning_enabled,
    )
    result = await Runner.run(
        calendar_agent,
        (
            'Check my calendar for the upcoming week and schedule workouts '
            'where my evening is free. '
            'My workout goal is to condition myself for a skiing season. '
            'Aim for a 1-hour 3-set program.'
        ),
        max_turns=30,
    )
    display(Markdown(result.final_output))

OPIK: Started logging traces to the "calendar-agent-1" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019c6747-de11-7d45-92e2-57caea737c9c&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.


I've analyzed your calendar for the upcoming week and scheduled a 7-day professional ski conditioning program. Each session is 1 hour long and follows a 3-set structure. 

I've placed the workouts in your free evening slots (generally at 7:00 PM), adjusting for your existing "Date with Uliana" on Thursday.

### **Your Workout Schedule:**
*   **Tue, Feb 17 (7:00 PM – 8:00 PM):** **Day 1: Max Integrated Strength** – Focus on heavy back squats and lower body stability.
*   **Wed, Feb 18 (7:00 PM – 8:00 PM):** **Day 2: Lateral Power & Explosiveness** – Skater jumps and lateral box jumps for rapid edge transitions.
*   **Thu, Feb 19 (5:30 PM – 6:30 PM):** **Day 3: Anaerobic Capacity (The "Ski Burn")** – High-intensity intervals and wall sits to prepare for long descents. *(Scheduled earlier due to your 7 PM date).*
*   **Fri, Feb 20 (7:00 PM – 8:00 PM):** **Day 4: Active Recovery** – Balance work on a Bosu ball and mobility to recover for the final push.
*   **Sat, Feb 21 (7:00 PM – 8:00 PM):** **Day 5: Eccentric Loading** – Tempo squats and Nordic curls to condition quads and protect your ACLs.
*   **Sun, Feb 22 (7:00 PM – 8:00 PM):** **Day 6: Agility & Reactive Coordination** – Ladder drills and depth jumps to mimic mogul skiing.
*   **Mon, Feb 23 (7:00 PM – 8:00 PM):** **Day 7: Alpine Simulation** – Incline treadmill work and a final high-rep circuit for endurance.

All workout details, including specific exercises and set counts, have been added to the description of each calendar event. Good luck with the training!

### Check if skips subagents if not needed

In [ ]:
async with get_calendar_full_access_server() as server:
    calendar_agent = Agent(
        name='calendar-agent',
        instructions=(
            'You are a helpful calendar assistant. '
            'Always use the available tools to answer. '
            'When you need information, you should call tools explicitly, using special syntax. '
            'Never guess arguments, only pass arguments that you\'re certain about. '
            'When you\'re getting an error, probably, you made a mistake in tool calling. '
            'Always be comprehensive and check yourself. '
            'Please note that if a function returns an empty list, '
            'this might be because there\'s something wrong with its arguments. '
            'Note that events may be scattered across different calendars.'
        ),
        model=litellm_model,
        mcp_servers=[server],
        tools=[
            workout_planner_agent.as_tool(
                tool_name='workout-planner',
                tool_description=(
                    'Plans a workout program, given user\'s goals '
                    'and number of days, as a professional coach. '
                    'Should be called to create workout plans.'
                ),
                parameters=WorkoutPlannerInput,
            )
        ],
        model_settings=low_reasoning_enabled,
    )
    result = await Runner.run(
        calendar_agent,
        'Check my calendar for the upcoming week and delete all planned workouts.',
        max_turns=30,
    )
    display(Markdown(result.final_output))

OK. I've deleted all 7 "Ski Prep Workout" events scheduled for the upcoming week from your primary calendar.

### Guardrails

In [12]:
class IsCalendarQueryOutput(BaseModel):
    is_calendar_query: bool = Field(description='Whether the query is related to user\'s calendar.')
    rejection_reason: str | None = Field(
        default=None, description='Rejection reason, if the query is not related to calendar.'
    )


guardrail_agent = Agent(
    name='guardrail-agent',
    instructions=(
        'You check whether user\'s query is related to user\'s calendar. '
        'You should reject all unrelated queries and requests.'
    ),
    model=light_model,
    output_type=IsCalendarQueryOutput,
)


@input_guardrail
async def calendar_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, input, context=ctx.context)
    assert isinstance(result.final_output, IsCalendarQueryOutput)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_calendar_query,
    )


def get_calendar_agent(server: MCPServerStdio):
    return Agent(
        name='calendar-agent',
        instructions=(
            'You are a helpful calendar assistant. '
            'Always use the available tools to answer. '
            'When you need information, you should call tools explicitly, using special syntax. '
            'Never guess arguments, only pass arguments that you\'re certain about. '
            'When you\'re getting an error, probably, you made a mistake in tool calling. '
            'Always be comprehensive and check yourself. '
            'Please note that if a function returns an empty list, '
            'this might be because there\'s something wrong with its arguments. '
            'Note that events may be scattered across different calendars.'
        ),
        model=litellm_model,
        mcp_servers=[server],
        tools=[
            workout_planner_agent.as_tool(
                tool_name='workout-planner',
                tool_description=(
                    'Plans a workout program, given user\'s goals '
                    'and number of days, as a professional coach. '
                    'Should be called to create workout plans.'
                ),
                parameters=WorkoutPlannerInput,
            )
        ],
        model_settings=low_reasoning_enabled,
        input_guardrails=[calendar_guardrail],
    )


async with get_calendar_full_access_server() as server:
    calendar_agent = get_calendar_agent(server)
    result = await Runner.run(
        calendar_agent,
        (
            'Check my calendar for the upcoming week and schedule a workout right before my date '
            'that we can do together with my gf, she loves running.'
        ),
        max_turns=30,
    )
    display(Markdown(result.final_output))

I've found your "Date with Uliana" scheduled for this Thursday, February 19th, from 7:00 PM to 10:00 PM.

To fit in a workout you can do together, I’ve scheduled a **75-minute running session** called "The Synchronized Surge" right before your date. 

**Event Details:**
*   **Workout:** Workout with GF (Running)
*   **Time:** Thursday, Feb 19th, 5:30 PM – 6:45 PM
*   **Description:** A professional coach-designed "Threshold Equalizer" session. It uses loop-based intervals so you can both run at your own intensity but finish each set together. It includes a warm-up, partner strength circuit (like plank high-fives), and a cooldown.

This gives you a 15-minute buffer before your date starts at 7:00 PM. Enjoy the run!

In [19]:
async with get_calendar_full_access_server() as server:
    calendar_agent = get_calendar_agent(server)
    try:
        result = await Runner.run(
            calendar_agent,
            'Who was the Emperor of Rome before Julius Caesar?',
            max_turns=30,
        )
        display(Markdown(result.final_output))
    except InputGuardrailTripwireTriggered as e:
        output = e.guardrail_result.output
        assert isinstance(output.output_info, IsCalendarQueryOutput)
        display(Markdown(output.output_info.rejection_reason))

This query is not related to the user's calendar. It is asking about historical information.